# 04 结局、时间与分割可用性核查

核查日期：2026-10-28。仅使用公开数据。此Notebook不拟合预测模型。

临床表中的空白坏死时间保持未知，后续MRI日期不等于该病灶无事件随访。掩膜非增强/坏死成分也不等于放射性坏死诊断。

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'src/rn_public').exists()
sys.path.insert(0,str(ROOT/'src'))
from rn_public.bundle import restore_sources
from rn_public.readiness import audit_outcomes
restore_sources(ROOT)
outcomes, counts = audit_outcomes(ROOT)
print(json.dumps(counts, indent=2))

{
  "clinical_lesions": 160,
  "patients": 75,
  "recorded_rn_times": 32,
  "unknown_rn": 128,
  "recorded_rn_after_index_srs": 30,
  "rn_at_or_before_index_srs": 2,
  "recorded_rn_no_index_srs": 0,
  "rn_after_death": 1,
  "index_srs_after_death": 2,
  "baseline_directory_candidates": 90,
  "strictly_pre_srs_directory_candidates": 90,
  "same_day_directory_candidates": 0,
  "baseline_existing_morphology_lesions": 21,
  "baseline_existing_morphology_rn_times": 1,
  "baseline_actual_mask_lesions": 53,
  "baseline_actual_mask_patients": 40,
  "baseline_actual_mask_rn_times": 17,
  "strict_baseline_existing_morphology_lesions": 21,
  "landmark90_directory_candidates": 41,
  "verified_non_events": 0,
  "final_eligible": 0
}


## 结局时间异常

所列记录需要澄清原始字段和治疗史。不能凭经验把异常时间改成合理数值。已记录事件时间也不表示逐例诊断依据已核实。

In [2]:
flags = outcomes[(outcomes.rn_after_death == 1) | (outcomes.index_srs_after_death == 1) | (outcomes.rn_evidence_class == 'rn_at_or_before_index_srs')]
print(flags[['lesion_key','first_srs_start_day','rn_time_day','death_day','rn_evidence_class']].to_string(index=False))
assert outcomes.loc[outcomes.rn_time_day.isna(),'rn_evidence_class'].eq('unknown').all()
assert outcomes.verified_non_event.sum() == 0
assert outcomes.final_eligible.sum() == 0

lesion_key  first_srs_start_day  rn_time_day  death_day           rn_evidence_class
  10024_L1                633.0        265.0       1169   rn_at_or_before_index_srs
  40114_L3               3784.0        582.0        645   rn_at_or_before_index_srs
  40114_L5               3784.0          NaN        645                     unknown
  40116_L1                  3.0       1685.0       1599 recorded_rn_after_index_srs


## 已有分割覆盖

全部NIfTI掩膜可用 `python scripts/audit_readiness.py` 重新检查。以下读取该脚本产生的逐文件和逐病灶结果。标签编号的病灶/成分解释通过体积核对获得，尚需原始编码说明确认；它们没有提供RN对照标签。

In [3]:
masks = pd.read_csv(ROOT/'data/processed/mask_file_audit.csv')
mask_lesions = pd.read_csv(ROOT/'data/processed/mask_lesion_audit.csv')
summary = json.loads((ROOT/'reports/readiness_audit.json').read_text())
print(json.dumps(summary['masks'],indent=2))
print('Baseline actual mask coverage:', int(outcomes.baseline_actual_mask_available.sum()))
print('Recorded RN times among those:', int(((outcomes.baseline_actual_mask_available == 1) & outcomes.rn_time_day.notna()).sum()))
print('Morphology dates alone undercount baseline coverage because of mixed date coding.')
assert len(masks) == summary['masks']['mask_files']
assert len(mask_lesions) == summary['masks']['mask_lesion_time_records']

{
  "mask_files": 373,
  "readable_mask_files": 373,
  "mask_lesion_time_records": 591,
  "mask_unique_lesions": 160,
  "morphology_unique_join_records": 495,
  "morphology_volume_agrees_1e_5_cm3": 336,
  "radiomics_unique_join_records": 591,
  "radiomics_volume_agrees_0_01_mm3": 512,
  "radiomics_volume_agrees_relative_0_1_percent": 591,
  "morphology_volume_agrees_relative_0_1_percent": 466,
  "morphology_year_code_2000_or_later": 97
}
Baseline actual mask coverage: 53
Recorded RN times among those: 17
Morphology dates alone undercount baseline coverage because of mixed date coding.


## DICOM抽样检查范围

每个选定目录按ZIP排列抽取首尾文件。检查CRC、可读性和基本头信息。这里的首尾不是按空间位置排序的首尾切片。未检查所有切片的空间完整性、像素、运动伪影、T1增强/FLAIR配准或目标病灶定位，因此不能标记为完整影像质控。

In [4]:
dicom = pd.read_csv(ROOT/'data/processed/dicom_sample_audit.csv')
print('Directories:',len(dicom),'sampled files:',int(dicom.sampled_files.sum()))
print(dicom.sample_read_status.value_counts().to_string())
print('Sample series UID consistent:',int(dicom.sample_same_series_uid.sum()))
print('Sample geometry consistent:',int(dicom.sample_same_geometry.sum()))
print('Full-series QC completed:',int(dicom.full_series_qc_complete.sum()))
assert dicom.full_series_qc_complete.sum() == 0

Directories: 469 sampled files: 938
sample_read_status
readable_headers    469
Sample series UID consistent: 468
Sample geometry consistent: 436
Full-series QC completed: 0


## 公开证据与当前结论

原始论文报告39个RN病灶，当前临床文件只有32个数值时间。没有找到可公开下载且能对应到本文件的补足标签表。后续鉴别研究不能替代治疗前风险研究的事件/非事件随访。完整结局和基线分割解决前，不输出AUC、风险率或临床效能。

In [5]:
evidence = json.loads((ROOT/'data/metadata/endpoint_evidence/reviewed_sources.json').read_text())
for source in evidence:
    print(source['source_id'],source['url'])
    print(source['endpoint_action'])
print('Modeling status:',summary['modeling_status'])

clinical_release https://api.figshare.com/v2/articles/22270933
Keep blanks unknown; do not infer a negative from follow-up dates or death.
descriptor_2023 https://pmc.ncbi.nlm.nih.gov/articles/PMC10104872/
Record the discrepancy; no extra seven events may be invented or assigned.
gomaa_2025_preprint https://arxiv.org/pdf/2511.18208
Do not treat an unpublished label crosswalk or a diagnostic selected cohort as a prospectively defined risk cohort.
rnd3_2026 https://www.frontiersin.org/journals/artificial-intelligence/articles/10.3389/frai.2026.1891883/full
Neither software inference nor the published cohort count can adjudicate this release's blank endpoints.
ocana_2026 https://pmc.ncbi.nlm.nih.gov/articles/PMC13452058/
Do not count this as a fully public downloadable validation cohort. No author contact has been made.
calculator https://radiationnecrosiscalculator.com/
Model probabilities must not be used as ground-truth labels.
Modeling status: not_ready_unresolved_outcome_labels_and_b